# Глава 2. Работа с текстовыми данными

In [1]:
import sys
print("Путь к Python:", sys.executable)
print("Версия Python:", sys.version)

Путь к Python: c:\Users\seera\AppData\Local\Python\pythoncore-3.14-64\python.exe
Версия Python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]


In [2]:
import sys
import subprocess

# Проверяем установленные пакеты
result = subprocess.run([sys.executable, "-m", "pip", "list"], capture_output=True, text=True)
print(result.stdout)

Package                 Version
----------------------- -----------
asttokens               3.0.1
certifi                 2026.4.22
charset-normalizer      3.4.7
colorama                0.4.6
comm                    0.2.3
debugpy                 1.8.20
decorator               5.2.1
executing               2.2.1
filelock                3.29.0
fsspec                  2026.3.0
idna                    3.13
ipykernel               7.2.0
ipython                 9.13.0
ipython_pygments_lexers 1.1.1
jedi                    0.19.2
Jinja2                  3.1.6
jupyter_client          8.8.0
jupyter_core            5.9.1
MarkupSafe              3.0.3
matplotlib-inline       0.2.1
mpmath                  1.3.0
nest-asyncio            1.6.0
networkx                3.6.1
numpy                   2.4.4
packaging               26.2
parso                   0.8.6
pip                     25.3
platformdirs            4.9.6
prompt_toolkit          3.0.52
psutil                  7.2.2
pure_eval              

In [3]:
import sys
import subprocess

# Установка torch и tiktoken
packages = ["torch", "tiktoken"]
for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

In [4]:
from importlib.metadata import version

print("Версия torch:", version("torch"))
print("Версия tiktoken:", version("tiktoken"))

Версия torch: 2.11.0
Версия tiktoken: 0.12.0


- Задача - подготовить данные и выборку образцов, чтобы "подготовить" входные данные для LLM

<img src="https://camo.githubusercontent.com/904aadb708a5c5d3ab94e96f2a89df661b677aee26f5292c8050261da68829ea/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30312e776562703f74696d657374616d703d31" width="800px">

## 2.1 Векторное представление слов

Существует множество форм вложения (эмбеддинг). Текущая задача - вложение текста

<img src="https://camo.githubusercontent.com/3e71d5547925bc40be3ad13ddebab7b8b39507f75af0a19db521d1e2e054fc4d/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30322e77656270" width="800px">

- LLM работают с вложениями в многомерные пространства (т.е. в тысячи измерений)
- Человек не может визуализировать такие многомерные пространства (люди мыслят в 1, 2 или 3 измерениях). На рисунке ниже показано 2-мерное пространство для вложений

<img src="https://camo.githubusercontent.com/2de90c75263eadcfe63a51e77da3f8cc86a3bbb98507e24de725629f82f20197/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30332e77656270" width="800px">

## 2.2 Токенизация текста

- Задача - разбить текст на более мелкие части, такие как отдельные слова и знаки препинания

<img src="https://camo.githubusercontent.com/937931bc39d9211a7d16803fcea8ce621422a635f72934af703468774e30bcd7/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30342e77656270" width="800px">

- Задача - загрузить необработанный текст для работы
- ["Вердикт" Эдит Уортон](https://en.wikisource.org/wiki/The_Verdict) - это рассказ, находящийся в открытом доступе

In [5]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    # Отправка GET-запрос по указанному URL с таймаутом 30 секунд
    # Если сервер не ответит за это время, запрос прервётся с ошибкой 
    response = requests.get(url, timeout=30) 

    # Проверка статус ответа
    # Если сервер вернул ошибку (4xx или 5xx), сразу выбрасывается исключение
    # Это быстрый способ проверить, что запрос успешен (2xx), без ручной проверки кода
    response.raise_for_status()

    # Открыть (или создаём) файл по указанному пути `file_path` для записи в бинарном режиме ("wb" — write binary)
    # Менеджер контекста `with` гарантирует автоматическое закрытие файла, даже если при записи возникнет ошибка
    # Файловый объект доступен через переменную `f`.
    with open(file_path, "wb") as f:
        f.write(response.content)

In [6]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Общее количество символов:", len(raw_text))
print(raw_text[:99])

Общее количество символов: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- Цель состоит в том, чтобы разметить и внедрить этот текст для LLM
- Задача - разработать простой разметчик на основе простого примера текста, который можно позже применить к тексту выше
- Следующее регулярное выражение будет разделяться пробелами

In [7]:
import re

text = "Hello, world. This, is a test."
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- Задача - разделить текст не только на пробелы, но и на запятые и точки

In [8]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- Задача - удалить пустые строки

In [9]:
# Удалить пробелы из каждого элемента, а затем отфильтровать все пустые строки
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- Задача - разобраться с другими типами знаков препинания, такими как точки, вопросительные знаки и так далее

In [10]:
text = "Hello, world. Is this-- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- Задача - применить текущую токенизацию к исходному тексту

<img src="https://camo.githubusercontent.com/322bfa4d8bce2fd2116543d2e8c8cb3732b87a0a676c55f8f959b33ab2c6c7c9/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30352e77656270" width="800px">

In [11]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- Задача - рассчитать общее количество токенов

In [12]:
print(len(preprocessed))

4690


## 2.3 Преобразование токенов в идентификаторы токенов

- Задача - пребразовать текстовые токены в идентификаторы токенов, которые позже можно обработать с помощью слоев встраивания

<img src="https://camo.githubusercontent.com/cbf2eaec29afc698a1fbe9839967d5fa07dfd07dc35482fa5298de03fc33af4b/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30362e77656270" width="800px">

- Задача - составить словарь из всех текущих уникальных токенов

In [13]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [14]:
vocab = {token:integer for integer,token in enumerate(all_words)}

- Ниже приведены первые 50 значений из словаря:

In [15]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- Ниже иллюстрация разбиения на символы короткого образца текста с использованием небольшого словарного запаса:

<img src="https://camo.githubusercontent.com/2f430af2ba9dce8e6b3fe730ec97235878c689db0c57003e5cb30eb1ab2778d8/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30372e776562703f313233" width="800px">

- Задача - создать класс-токенизатор и добавить туда текущий словарь

In [16]:
class SimpleTokenizerV1:

    # Конструктор класса
    # Вызывается при создании объекта
    # self - ссылка на сам объект
    # Принимает словарь vocab, где ключ — слово, значение — его ID.
    def __init__(self, vocab):

        # Сохранение словаря для прямого преобразования: строка → число
        self.str_to_int = vocab

        # Создание обратного словаря (число → строка) с помощью генератора: для каждой пары s,i в vocab.items() создать запись {i: s}
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
                                
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])

        # Замена пробелов перед указанными знаками препинания
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)

        return text

- Функция "encode" преобразует текст в идентификаторы токенов
- Функция "decode" преобразует идентификаторы токенов обратно в текст

<img src="https://camo.githubusercontent.com/343b55a9079b960142914b9b1c795535668caeb8facd8fe48bd8c36bbd7162f3/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30382e776562703f313233" width="800px">

- Можно использовать токенизатор для кодирования (то есть токенизации) текстов в целые числа
- Эти целые числа затем могут быть встроены (позже) в качестве входных данных для LLM

In [17]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- Можно декодировать целые числа обратно в текст

In [18]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [19]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Добавление специальных контекстных токенов

- Полезно добавить несколько "специальных" маркеров для неизвестных слов и обозначить конец текста

<img src="https://camo.githubusercontent.com/94694d0039ff3000020a8d0c1847c55d859b0e22dd82d377414473b6370228b9/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f30392e776562703f313233" width="800px">

- Некоторые токенизаторы используют специальные токены, чтобы помочь LLM с дополнительным контекстом
- Некоторые из этих специальных токенов
 - `[BOS]` (начало последовательности) обозначает начало текста
 - `[EOS]` (конец последовательности) обозначает место, где заканчивается текст (обычно используется для объединения нескольких несвязанных текстов, например, двух разных статей в Википедии или двух разных книг и так далее).
 - `[PADDING]` (заполнение), если выполняется обучаем LLM с размером пакета больше 1 (мы можем включать несколько текстов разной длины; с помощью маркера padding мы дополняем короткие тексты до максимально длинных, чтобы все тексты имели одинаковую длину).
- `[UNK]` для обозначения слов, которые не включены в словарный запас

</br>

- GPT-2 не нуждается ни в одном из этих токенов, упомянутых выше, а использует только токен `<|endoftext|>` для уменьшения сложности
- Токен `<|endoftext|>` аналогичен токену `[EOS]`, упомянутому выше
- GPT также использует `<|endoftext|>` для заполнения (поскольку обычно используется маска при обучении на пакетных входных данных, в любом случае заполненные токены не используются, поэтому не имеет значения, что это за токены).
- GPT-2 не использует маркер `<UNK>` для слов, которые не входят в словарный запас; вместо этого в GPT-2 используется маркер для кодирования пар байтов (BPE), который разбивает слова на подслова

- Задача - использовать токены "<|endoftext|>" между двумя независимыми источниками текста:

<img src="https://camo.githubusercontent.com/f21e09f3fc21526bdeae1ba6e4e468eacc827407f75a00fb003888038c4ba4dd/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31302e77656270" width="800px">

- Ввод текста со словами, которых нет в текущем словаре:

In [20]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

- Приведенное выше сообщение приводит к ошибке, поскольку слово "Hello" не содержится в словаре
- Чтобы справиться с такими случаями, нужно добавить в словарь специальные лексемы, такие как `<|unk|>`, для обозначения неизвестных слов
- Поскольку идет расширение словарного запаса, добавляется еще один токен под названием `<|endoftext|>`, который используется в обучении GPT-2 для обозначения конца текста (и он также используется между объединенными текстами, например, если обучающие наборы данных состоят из нескольких статей, книг и т.д.)

In [21]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [22]:
len(vocab.items())

1132

In [23]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- Необходимо соответствующим образом настроить токенизатор, чтобы он знал, когда и как использовать новый токен `<unk>`

In [24]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Замена пробелов перед указанными знаками препинания
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Размещение текст с помощью модифицированного токенизатора:

In [25]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [26]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [27]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 Кодирование пар байтов

- В GPT-2 в качестве токенизатора используется кодировка BytePair (BPE)
- это позволяет модели разбивать слова, которых нет в ее предопределенном словаре, на более мелкие подслова или даже отдельные символы, что позволяет ей обрабатывать слова, которых нет в словаре
- Например, если в словаре GPT-2 нет слова "незнакомое слово", оно может обозначать его как ["незнакомый", "подвздошный", "слово"] или как-то иначе, в зависимости от его обученных слияний BPE
- Оригинальный токенизатор BPE можно найти здесь: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- В текущей задаче используется токенизатор BPE из библиотеки OpenAI с открытым исходным кодом [tiktoken](https://github.com/openai/tiktoken), которая реализует свои основные алгоритмы в Rust для повышения производительности вычислений

Оригинальный токенизатор BPE:

In [28]:
%%script false --no-raise-error
# Это заставит Jupyter попытаться выполнить код как программу с именем false
# Такой программы нет, но из-за флага --no-raise-error ошибка не появится, и ячейка просто "тихо" пропустится

"""Byte pair encoding utilities"""

import os
import json
import regex as re
from functools import lru_cache

# Проблема: BPE работает с символами Unicode
# Если добавить в словарь все возможные Unicode-символы (а их больше 143,000), словарь станет огромным
# Можно хранить текст байтово (в UTF-8), где значений байтов всего 256 (от 0 до 255)
# Однако многие из них — это управляющие символы (например, перенос строки), с которыми BPE не умеет работать
# Решение функции: Создаёт словарь-переводчик между байтами и "безопасными" Unicode-символами
# Как она работает:
## Берёт видимые символы из таблицы Unicode: от ! до ~ (94 символа), от ¡ до ¬ и от ® до ÿ (это 162 символа). Всего 256 "хороших" символов
## Создаёт первую часть таблицы, где байты от 33 до 126 и другие напрямую отображаются в эти же символы (байт 65 → 'A')
## Для оставшихся байтов (от 128 до 255, которые могут быть управляющими) создаёт новые безопасные символы из неиспользованных номеров Unicode (начиная с 256)
## Возвращает словарь, где каждому байту (0–255) соответствует безопасный Unicode-символ.
# Итог: После такой обработки можно представлять любой текст как последовательность безопасных символов, которые BPE "съест" без проблем
@lru_cache()
def bytes_to_unicode():
    """
    Returns list of utf-8 byte and a corresponding list of unicode strings.
    The reversible bpe codes work on unicode strings.
    This means you need a large # of unicode characters in your vocab if you want to avoid UNKs.
    When you're at something like a 10B token dataset you end up needing around 5K for decent coverage.
    This is a signficant percentage of your normal, say, 32K bpe vocab.
    To avoid that, we want lookup tables between utf-8 bytes and unicode strings.
    And avoids mapping to whitespace/control characters the bpe code barfs on.
    """
    bs = list(range(ord("!"), ord("~")+1))+list(range(ord("¡"), ord("¬")+1))+list(range(ord("®"), ord("ÿ")+1))
    cs = bs[:]
    n = 0
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8+n)
            n += 1
    cs = [chr(n) for n in cs]
    return dict(zip(bs, cs))

# Очень простая функция: берёт слово как кортеж символов (например, ('к', 'о', 'т')) и возвращает все пары соседних символов: {('к', 'о'), ('о', 'т')}
# Зачем: Чтобы BPE мог найти, какая пара символов встречается чаще всего и заменить её на один новый символ
def get_pairs(word):
    """Return set of symbol pairs in a word.

    Word is represented as tuple of symbols (symbols being variable-length strings).
    """
    pairs = set()
    prev_char = word[0]
    for char in word[1:]:
        pairs.add((prev_char, char))
        prev_char = char
    return pairs

# Это сердце BPE:
## __init__ (инициализация)
## encoder: готовый словарь токен → число (например, "кот" → 1234).
## bpe_merges: история слияний пар
### Это список пар символов, которые объединялись, в порядке приоритета. Например: [('к', 'о'), ('ко', 'т'), ...]
### Чем раньше в списке, тем приоритетнее слияние
### Создаёт обратные словари (decoder, byte_decoder) для перевода в обратную сторону
# bpe_ranks: присваивает каждой паре ранг (0 — самая приоритетная)
# cache: хранит результаты разбиения, чтобы не считать одно и то же слово дважды
# pat: регулярное выражение, которое разбивает текст на слова и знаки пунктуации. Пример: "Hello, world!" → ["Hello", ",", "world", "!"]
class Encoder:
    def __init__(self, encoder, bpe_merges, errors='replace'):
        self.encoder = encoder
        self.decoder = {v:k for k,v in self.encoder.items()}
        self.errors = errors # how to handle errors in decoding
        self.byte_encoder = bytes_to_unicode()
        self.byte_decoder = {v:k for k, v in self.byte_encoder.items()}
        self.bpe_ranks = dict(zip(bpe_merges, range(len(bpe_merges))))
        self.cache = {}

        # Следовало бы добавить re.IGNORECASE , чтобы слияния BPE могли происходить для версий сокращений с заглавной буквы
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

    # Это самая важная функция. Она разбивает слово на части по методу BPE
    # Как работает (на примере слова low):
    ## Допустим, слово — "low"
    ## Порядок пар (из истории слияний): ('l','o'), ('lo','w') — самый приоритетный, потом ('o','w').
    ## Представляем слово как ('l', 'o', 'w'). Находим все пары: ('l','o') и ('o','w').
    ## Ищем приоритетную пару — это ('l','o'), потому что у неё ранг меньше.
    ## Объединяем эту пару в новом слове: ('lo', 'w').
    ## Снова ищем пары — теперь только ('lo', 'w'). Есть она в истории слияний? Да! Объединяем: ('low',).
    ## Слово теперь из одного элемента — стоп. Итог: "low" так и осталось целым словом.
    # А вот если бы пары ('lo', 'w') не было в истории, слово разбилось бы как 'lo', 'w'.
    def bpe(self, token):
        if token in self.cache:
            return self.cache[token]

        # Превращение строки в кортеж символов
        # Это нужно, чтобы можно было искать индексы и работать с соседними элементами
        # Пример: "low" → ('l', 'o', 'w').
        word = tuple(token)

        # get_pairs() возвращает множество всех соседних пар в слове
        # Для ('l', 'o', 'w') это будет {('l','o'), ('o','w')
        pairs = get_pairs(word)

        if not pairs:
            return token

        while True:

            # Нахождение пары с наименьшим рангом (то есть самой приоритетной для слияния)
            # Ранг хранится в self.bpe_ranks
            # Если такой пары в ранках нет, то get вернёт float('inf') (бесконечность) — эта пара никогда не будет выбрана.
            bigram = min(pairs, key = lambda pair: self.bpe_ranks.get(pair, float('inf')))

            if bigram not in self.bpe_ranks:
                break
            first, second = bigram
            new_word = []
            i = 0
            while i < len(word):
                try:
                    j = word.index(first, i)
                    new_word.extend(word[i:j])
                    i = j
                except:
                    new_word.extend(word[i:])
                    break

                if word[i] == first and i < len(word)-1 and word[i+1] == second:
                    new_word.append(first+second)
                    i += 2
                else:
                    new_word.append(word[i])
                    i += 1
            new_word = tuple(new_word)
            word = new_word
            if len(word) == 1:
                break
            else:
                pairs = get_pairs(word)
        word = ' '.join(word)
        self.cache[token] = word
        return word

# Прямой перевод текста в последовательность чисел (токенов):
## Разбивает текст на слова токенизатором pat
## Каждое слово кодирует в байты, а потом заменяет байты на безопасные Unicode-символы (из bytes_to_unicode)
## Применяет BPE, чтобы разбить слово на подслова.
## Каждое подслово заменяет на его числовой код из encoder
## Возвращает список чисел.
    def encode(self, text):
        bpe_tokens = []
        for token in re.findall(self.pat, text):
            token = ''.join(self.byte_encoder[b] for b in token.encode('utf-8'))
            bpe_tokens.extend(self.encoder[bpe_token] for bpe_token in self.bpe(token).split(' '))
        return bpe_tokens

# Обратный процесс:
## Переводит числа в безопасные Unicode-символы через decoder
## Превращает безопасные символы обратно в байты через byte_decoder
## Декодирует байты в читаемый текст в кодировке UTF-8
    def decode(self, tokens):
        text = ''.join([self.decoder[token] for token in tokens])
        text = bytearray([self.byte_decoder[c] for c in text]).decode('utf-8', errors=self.errors)
        return text

# Просто загружает готовые файлы (словарь и историю слияний) из папки с моделью и возвращает готовый объект Encoder
def get_encoder(model_name, models_dir):
    with open(os.path.join(models_dir, model_name, 'encoder.json'), 'r') as f:
        encoder = json.load(f)
    with open(os.path.join(models_dir, model_name, 'vocab.bpe'), 'r', encoding="utf-8") as f:
        bpe_data = f.read()
    bpe_merges = [tuple(merge_str.split()) for merge_str in bpe_data.split('\n')[1:-1]]
    return Encoder(
        encoder=encoder,
        bpe_merges=bpe_merges,
    )

Couldn't find program: 'false'


In [29]:
import importlib
import tiktoken

print("Версия tiktoken:", importlib.metadata.version("tiktoken"))

Версия tiktoken: 0.12.0


In [30]:
tokenizer = tiktoken.get_encoding("gpt2")

In [31]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [32]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- Токенизаторы BPE разбивают неизвестные слова на подзаголовки и отдельные символы:

<img src="https://camo.githubusercontent.com/b3791f38778406e8894465a01617788f75989f06e8ac2685fec8378d8d60ea75/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31312e77656270" width="800px">

### Упражнение 2.1. Кодирование неизвестных слов с помощью пар байтов

In [33]:
tokenizer = tiktoken.get_encoding("gpt2")
integers = tokenizer.encode("Akwirw ier")
print(integers)

[33901, 86, 343, 86, 220, 959]


In [34]:
for i in integers:
    print(f"{i} -> {tokenizer.decode([i])}")

33901 -> Ak
86 -> w
343 -> ir
86 -> w
220 ->  
959 -> ier


In [35]:
tokenizer.decode([33901, 86, 343, 86, 220, 959])

'Akwirw ier'

## 2.6 Выборка данных с помощью скользящего окна

- Обучение LLM заключается в генерации по одному слову за раз, поэтому нужно подготовить обучающие данные, в которых следующее слово в последовательности представляет цель для прогнозирования:

<img src="https://camo.githubusercontent.com/d29cea0621158ecea28778582d1ca8f2fbae94a85808d758cd0930fa2587d88a/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31322e77656270" width="800px">

In [36]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- Для каждого фрагмента текста нужны входные данные и целевые объекты
- Чтобы модель предсказывала следующее слово, целевыми объектами являются входные данные, сдвинутые на одну позицию вправо

In [37]:
enc_sample = enc_text[50:]

In [38]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- Один за другим предсказания выглядели бы следующим образом:

In [39]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [40]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- Задача - установить и импортировать PyTorch. Реализовать простой загрузчик данных, который выполняет итерацию по входному набору данных и возвращает исходные данные и целевые объекты, сдвинутые на единицу

Словил предупреждение:

`c:\Users\seera\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
cpu = _conversion_method_template(device=torch.device("cpu"))
`

PyTorch работает и без NumPy, просто ограниченно в конвертации тензоров в numpy-массивы (.numpy() не сработает, пока не установите NumPy).

In [41]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
import torch
print("Версия PyTorch:", torch.__version__)

Версия PyTorch: 2.11.0+cpu


- Мы используем метод скользящего окна, изменяя положение на +1:

<img src="https://camo.githubusercontent.com/3b88416514e89a7fe6b3043bb8afd6b8f9a6a843269038fc6af5e5cd025b6c99/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31332e776562703f313233" width="800px">

- Задача - создать набор данных и загрузчик данных, которые извлекают фрагменты из входного текстового набора данных

In [43]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Токенизация всего текста
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Использование скользящего окна, чтобы разбить книгу на перекрывающиеся последовательности максимальной длины
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [44]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):

    # Инициализация токенайзера
    tokenizer = tiktoken.get_encoding("gpt2")

    # Создание набора данных
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Создание загрузчика данных
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

- Тестирования загрузчика данных с размером пакета 1 для LLM с размером контекста 4:

In [45]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [46]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [47]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- Пример использования шага, равного длине контекста (здесь: 4), как показано ниже:

<img src="https://camo.githubusercontent.com/03d3b442d9d53e70086fbc11a4c0cad65211a76dd11eedf40552341c0d57676e/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31342e77656270" width="800px">

- Можно создавать пакетные выходные данные
- Увеличение шага делается, чтобы избежать совпадений между пакетами, поскольку большее количество совпадений может привести к переобучению

In [48]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Входные данные:\n", inputs)
print("\nЦель:\n", targets)

Входные данные:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Цель:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Упражнение 2.2. Загрузчики данных с разными значениями шага смещения и размера контекста

In [49]:
def create_dataloader(txt, batch_size=4, max_length=256, stride=128):

    # Инициализация токенайзера
    tokenizer = tiktoken.get_encoding("gpt2")

    # Создание набора данных
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Создание загрузчика данных
    dataloader = DataLoader(dataset, batch_size=batch_size)

    return dataloader


with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(raw_text)

In [50]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=2, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

tensor([[  40,  367],
        [2885, 1464],
        [1807, 3619],
        [ 402,  271]])

In [51]:
dataloader = create_dataloader(raw_text, batch_size=4, max_length=8, stride=2)

for batch in dataloader:
    x, y = batch
    break

x

tensor([[   40,   367,  2885,  1464,  1807,  3619,   402,   271],
        [ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138],
        [ 1807,  3619,   402,   271, 10899,  2138,   257,  7026],
        [  402,   271, 10899,  2138,   257,  7026, 15632,   438]])

## 2.7 Создание векторных пердставлений токенов

- Задача - внедрить токены в непрерывное векторное представление, используя слой внедрения
- Обычно эти слои внедрения являются частью самого LLM и обновляются (обучаются) во время обучения модели

<img src="https://camo.githubusercontent.com/c987cbe5dd843f4d217b7e147bdeff485c88e9825711861abd0c02c80f5331d9/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31352e77656270" width="800px">

- Дано: четыре примера ввода с входными идентификаторами 2, 3, 5 и 1 (после токенизации):

In [52]:
input_ids = torch.tensor([2, 3, 5, 1])

- Пусть имеется небольшой словарный запас, состоящий всего из 6 слов. Задача - создать вложения размером 3:

In [53]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- В результате получилась бы весовая матрица размером 6х3:

In [54]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


`nn.Embedding(vocab_size, output_dim)` — это по сути **таблица поиска (lookup table)**. Представьте себе матрицу размером `[vocab_size × output_dim]`, где:
- Каждая **строка** — это векторное представление (эмбеддинг) одного токена/слова
- Индекс строки соответствует ID токена (от 0 до vocab_size-1)

Почему именно эти числа?

Эти числа — **случайные**!

При создании `nn.Embedding` веса инициализируются случайным образом из **нормального распределения** (среднее ≈ 0, стандартное отклонение = 1). Поэтому числа выглядят хаотично:

```python
embedding_layer = torch.nn.Embedding(vocab_size=6, output_dim=3)
```

Что произошло:
1. PyTorch создал матрицу 6×3 (6 токенов, каждый размерности 3)
2. Заполнил её случайными числами из `N(0,1)`
3. Включил `requires_grad=True` — значит, эти числа будут **обновляться во время обучения**

Как это работает на практике?

```python
# Допустим, есть индексы токенов
input_indices = torch.tensor([0, 2, 4])  # слова с ID 0, 2, 4

# Embedding слой просто "достаёт" нужные строки из матрицы
embeddings = embedding_layer(input_indices)
# Результат:
# tensor([[ 0.3374, -0.1778, -0.1690],   ← строка 0
#         [ 1.2753, -0.2010, -0.1606],   ← строка 2
#         [-1.1589,  0.3255, -0.6315]])  ← строка 4
```

**Никаких вычислений!** Только извлечение нужных строк по индексам

Почему случайная инициализация?

Потому что на старте обучения нет информации о смысле слов. В процессе обучения (например, word2vec или при тренировке нейросети) эти числа будут меняться так, чтобы:
- Похожие по смыслу слова имели близкие векторы
- Выполнялась целевая задача (классификация, генерация текста и т.д.)

Визуализация:

Матрица эмбеддингов (6 слов × 3 признака):
```
       dim0    dim1    dim2
ID0  [ 0.34,  -0.18,  -0.17]
ID1  [ 0.92,   1.58,   1.30]
ID2  [ 1.28,  -0.20,  -0.16]
ID3  [-0.40,   0.97,  -1.15]
ID4  [-1.16,   0.33,  -0.63]
ID5  [-2.84,  -0.78,  -1.41]
```

Когда будет запрос ID2 — будет получен вектор `[1.28, -0.20, -0.16]`

После обучения веса перестанут быть случайными — они будут отражать семантические связи между словами

- Задача - преобразовать токен с идентификатором 3 в трехмерный вектор:

In [55]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- Приведенное выше значение является 4-й строкой весовой матрицы `embedding_layer`
- Задача - вставить все четыре значения `input_ids`, приведенные выше:

In [56]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- Слой внедрения - это, по сути, операция поиска:

<img src="https://camo.githubusercontent.com/6830f9f9f2fa4e11daa07d25a8dca7af532b1a8c743156c710f62e244daa39e7/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31362e776562703f313233" width="800px">

## 2.8 Кодирование позиций слов

- Слой встраивания преобразует идентификаторы в идентичные векторные представления независимо от того, где они расположены во входной последовательности:

<img src="https://camo.githubusercontent.com/90155d50e064bd2e897687fc3cabc58f4ec2c0f0310ff2a871af5a2f6fa5f8fd/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31372e77656270" width="800px">

- Позиционные вложения объединяются с вектором вложения токена для формирования входных вложений для большой языковой модели:

<img src="https://camo.githubusercontent.com/09d3d00572e7a68f2184f08f42a1e8b221728ab4e8907384f16ebd8399fd0a1e/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31382e77656270" width="800px">

- Кодер BytePair имеет словарный запас размером 50 257
- Задача - закодировать входные токены в 256-мерное векторное представление:

In [57]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- Если выборка данных берется из dataloader, токены вставляются в каждом пакете в 256-мерный вектор
- Если размер пакета 8 и в каждом по 4 токена, это приводит к тензору размером 8 x 4 x 256:

In [58]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [59]:
print("Идентификаторы токенов:\n", inputs)
print("\nФорма входных данных:\n", inputs.shape)

Идентификаторы токенов:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Форма входных данных:
 torch.Size([8, 4])


In [60]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

print(token_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 0.4913,  1.1239,  1.4588,  ..., -0.3995, -1.8735, -0.1445],
         [ 0.4481,  0.2536, -0.2655,  ...,  0.4997, -1.1991, -1.1844],
         [-0.2507, -0.0546,  0.6687,  ...,  0.9618,  2.3737, -0.0528],
         [ 0.9457,  0.8657,  1.6191,  ..., -0.4544, -0.7460,  0.3483]],

        [[ 1.5460,  1.7368, -0.7848,  ..., -0.1004,  0.8584, -0.3421],
         [-1.8622, -0.1914, -0.3812,  ...,  1.1220, -0.3496,  0.6091],
         [ 1.9847, -0.6483, -0.1415,  ..., -0.3841, -0.9355,  1.4478],
         [ 0.9647,  1.2974, -1.6207,  ...,  1.1463,  1.5797,  0.3969]],

        [[-0.7713,  0.6572,  0.1663,  ..., -0.8044,  0.0542,  0.7426],
         [ 0.8046,  0.5047,  1.2922,  ...,  1.4648,  0.4097,  0.3205],
         [ 0.0795, -1.7636,  0.5750,  ...,  2.1823,  1.8231, -0.3635],
         [ 0.4267, -0.0647,  0.5686,  ..., -0.5209,  1.3065,  0.8473]],

        ...,

        [[-1.6156,  0.9610, -2.6437,  ..., -0.9645,  1.0888,  1.6383],
         [-0.3985, -0.9235, -1.31

- GPT-2 использует встраивание абсолютных позиций. Задача - создать еще один слой встраивания:

In [61]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

print(pos_embedding_layer.weight)

Parameter containing:
tensor([[ 1.7375, -0.5620, -0.6303,  ..., -0.2277,  1.5748,  1.0345],
        [ 1.6423, -0.7201,  0.2062,  ...,  0.4118,  0.1498, -0.4628],
        [-0.4651, -0.7757,  0.5806,  ...,  1.4335, -0.4963,  0.8579],
        [-0.6754, -0.4628,  1.4323,  ...,  0.8139, -0.7088,  0.4827]],
       requires_grad=True)


In [62]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

print(pos_embeddings)

torch.Size([4, 256])
tensor([[ 1.7375, -0.5620, -0.6303,  ..., -0.2277,  1.5748,  1.0345],
        [ 1.6423, -0.7201,  0.2062,  ...,  0.4118,  0.1498, -0.4628],
        [-0.4651, -0.7757,  0.5806,  ...,  1.4335, -0.4963,  0.8579],
        [-0.6754, -0.4628,  1.4323,  ...,  0.8139, -0.7088,  0.4827]],
       grad_fn=<EmbeddingBackward0>)


- Задача - добавить токен и позиционные вложения, чтобы создать входные вложения, используемые в LLM:

In [63]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

print(input_embeddings)

torch.Size([8, 4, 256])
tensor([[[ 2.2288,  0.5619,  0.8286,  ..., -0.6272, -0.2987,  0.8900],
         [ 2.0903, -0.4664, -0.0593,  ...,  0.9115, -1.0493, -1.6473],
         [-0.7158, -0.8304,  1.2494,  ...,  2.3952,  1.8773,  0.8051],
         [ 0.2703,  0.4029,  3.0514,  ...,  0.3595, -1.4548,  0.8310]],

        [[ 3.2835,  1.1749, -1.4150,  ..., -0.3281,  2.4332,  0.6924],
         [-0.2199, -0.9114, -0.1750,  ...,  1.5337, -0.1998,  0.1462],
         [ 1.5197, -1.4240,  0.4391,  ...,  1.0494, -1.4318,  2.3057],
         [ 0.2893,  0.8346, -0.1884,  ...,  1.9602,  0.8709,  0.8796]],

        [[ 0.9662,  0.0952, -0.4640,  ..., -1.0320,  1.6290,  1.7771],
         [ 2.4468, -0.2154,  1.4984,  ...,  1.8766,  0.5595, -0.1423],
         [-0.3856, -2.5393,  1.1556,  ...,  3.6157,  1.3267,  0.4944],
         [-0.2487, -0.5275,  2.0009,  ...,  0.2930,  0.5977,  1.3300]],

        ...,

        [[ 0.1219,  0.3991, -3.2740,  ..., -1.1921,  2.6637,  2.6728],
         [ 1.2438, -1.6436, -1.11

- На начальном этапе процесса обработки входных данных вводимый текст сегментируется на отдельные токены
- После такой сегментации эти токены преобразуются в идентификаторы токенов на основе предопределенного словаря:

<img src="https://camo.githubusercontent.com/2a77e1836de72cc6a8c806b45873d0418c958f8c3c383b5dac3bad31d221db69/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830325f636f6d707265737365642f31392e77656270" width="800px">